In [16]:
"""
Created on Wed Jul  7 11:54:11 2021
Updated last: 22/01/2023
@author: Raphael Bendahan-West

description: Get metadata + spectra/wavelength from fits files 
"""
import pandas as pd
import numpy as np
from astropy.io import fits
import sys, os


In [ ]:

class GetSpectra():
    
    CaIIK = 3933.66 # Ca II K line in Angstrom
    CaIIH = 3968.47 # Ca II H line in Angstrom
    
    def __init__(self, nb_files = None, length_spec = None, loaded_sK = None, loaded_sH = None):
        
        # Option to ammend downloaded spectra by loading the spectrum arrays
        if loaded_sK is not None and loaded_sH is not None:
            self.sK = loaded_sK
            self.sH = loaded_sH

            # prevents wavelengths to be calculated/loaded
            self.wave_initK = True 
            self.wave_initH = True 


        # Initialising arrays only if length is specified
        if length_spec is not None and nb_files is not None:
            self.sK = np.zeros([nb_files, length_spec])
            self.sH = np.zeros([nb_files, length_spec])

            self.wK = np.zeros(length_spec)
            self.wH = np.zeros(length_spec)

            self.rvK = np.zeros(length_spec)
            self.rvH = np.zeros(length_spec)

            # Flag when wavelength+rv have been calculated and stored
            self.wave_initK = False # Same arrays for every spectrum
            self.wave_initH = False # Same arrays for every spectrum

        self.corrupt = pd.DataFrame({'File': pd.Series(dtype='object')})
        self.corrupt_s = pd.DataFrame({'File': pd.Series(dtype='object')})

    # def clean_data(self):
    #     ''' Clean the data from corrupt files'''
    #     return
    
    def save_arr(self, s_path, w_path):
        self.corrupt.to_pickle(s_path + 'corrupt.pkl')
        np.save(s_path + 'sK.npy', self.sK)
        np.save(s_path + 'sH.npy', self.sH)

        if os.path.exists(w_path) == False:
            os.mkdir(w_path)
        
        np.save(w_path + 'wK.npy', self.wK)
        np.save(w_path + 'rvK.npy', self.rvK)

        np.save(w_path + 'wH.npy', self.wH)
        np.save(w_path + 'rvH.npy', self.rvH)

    def update_archive(self, index, file, keys, df):
        lines = [self.CaIIH, self.CaIIK]
        try:
            with fits.open(file) as h:
                for line in lines:
                    self.get_spec(file, line, index)

                vals = self.get_meta(h, keys)

                updated_df = pd.DataFrame(np.insert(df.values, index, values=vals, axis=0))
                updated_df.columns = df.columns

        except:
            vals = [np.nan] * len(keys)
            checked = True
            vals.append(checked)

            updated_df = pd.DataFrame(np.insert(df.values, index, values=vals, axis=0))
            updated_df.columns = df.columns

            self.sK = np.insert(self.sK, index, np.zeros(2000), axis=0)
            self.sH = np.insert(self.sH, index, np.zeros(2000), axis=0)

            self.corrupt.loc[index, 'File'] = file

        return updated_df

    def _extract_spectrum_triplet(self, file):
        data = fits.getdata(file)
        names = getattr(getattr(data, 'dtype', None), 'names', None)

        if names is not None:
            row = data[0]
            if {'WAVE', 'FLUX_REDUCED', 'ERR_REDUCED'}.issubset(names):
                return row['WAVE'], row['FLUX_REDUCED'], row['ERR_REDUCED']
            if {'WAVE', 'FLUX', 'ERR'}.issubset(names):
                return row['WAVE'], row['FLUX'], row['ERR']
            return row[0], row[1], row[2]

        row = data[0]
        return row[0], row[1], row[2]

    def get_spec(self, file, wave, idx):

        wavelength, spec, error = self._extract_spectrum_triplet(file)
        condition = np.where(np.round(wavelength, 2) == wave)[0][0] # to get actual value

        if wave == self.CaIIK:
            self.sK = np.insert(self.sK, idx, spec[condition-1000:condition+1000], axis=0)

        elif wave == self.CaIIH:
            self.sH = np.insert(self.sH, idx, spec[condition-1000:condition+1000], axis=0)

        else:
            print('Wrong line')
            sys.exit()

        return

    def get_meta(self, h, keys):
        header = h[0].header
        val = []

        # Read only the keys requested by the caller; no legacy header transforms.
        for k in keys:
            val.append(header.get(k, np.nan))

        checked = True
        val.append(checked)

        return val
        

    def get_spectrum(self, index, file):
        ''' Update function by creating new function that does metadata + spectra
        Only opens the file once and not for both'''
        lines = [self.CaIIH, self.CaIIK]
        print(f'processing file: {file}')

        try:
            with fits.open(file) as h:
                header = h[0].header
                wavelmin = header.get('WAVELMIN')
                wavelmax = header.get('WAVELMAX')

                if wavelmin is None or wavelmax is None:
                    raise KeyError('Missing WAVELMIN/WAVELMAX in FITS header')

                wavelength, spec, error = self._extract_spectrum_triplet(file)

                if len(wavelength) == 0:
                    raise ValueError('Empty wavelength array')

                header_use_nm = (wavelmax is not None) and (wavelmax < 1000)
                wave_use_nm = np.nanmedian(wavelength) < 1000
                print(f'wavelength range: WAVELMIN={wavelmin}, WAVELMAX={wavelmax}')

                for line in lines:
                    target_header = line / 10.0 if header_use_nm else line
                    target_wave = line / 10.0 if wave_use_nm else line

                    if wavelmin < target_header < wavelmax:
                        idx = np.where(np.isclose(wavelength, target_wave, atol=0.02))[0]

                        if idx.size == 0:
                            print(f'error c: line target {target_wave} not found in wavelength grid')
                            self.corrupt_s.loc[index, 'File'] = file
                            continue

                        condition = int(idx[0])

                        if condition < 1000 or condition + 1000 > len(spec):
                            print('error d: line too close to spectrum boundary for 2000-point cutout')
                            self.corrupt_s.loc[index, 'File'] = file
                            continue

                        if line == self.CaIIK:
                            self.sK[index,:] = spec[condition-1000:condition+1000]

                            if self.wave_initK is False:
                                self.wK = wavelength[condition-1000:condition+1000]
                                self.rvK = 3e5 * (self.wK-target_wave)/(target_wave)

                                self.wave_initK = True

                        elif line == self.CaIIH:
                            self.sH[index,:] = spec[condition-1000:condition+1000]

                            if self.wave_initH is False:
                                self.wH = wavelength[condition-1000:condition+1000]
                                self.rvH = 3e5 * (self.wH-target_wave)/(target_wave)

                                self.wave_initH = True

                        else:
                            print('Wrong line')
                            sys.exit()

                    else:
                        print('error a')
                        self.corrupt_s.loc[index, 'File'] = file

        except Exception as e:
            print(f'error b: {type(e).__name__}: {e}')
            print(file)
            self.corrupt_s.loc[index, 'File'] = file

        return

    def pads(self, s1):
        if len(s1) < 6:
            pad = '0' * ( 6 - len(s1) )
            s1 = pad + s1
        return s1

    def get_metadata(self, index, file, keys, main_df):
       
        try:
            with fits.open(file) as h:
                header = h[0].header
                val = []

                # Read only the keys requested by the caller; no legacy header transforms.
                for k in keys:
                    val.append(header.get(k, np.nan))

                checked = True
                val.append(checked)

                main_df.loc[index] = val

        except:
            val = [np.nan] * len(keys)
            checked = True
            val.append(checked)
            main_df.loc[index] = val
            self.corrupt.loc[index, 'File'] = file
        
        return main_df